In [1]:
import pandas as pd
import numpy as np

model_df = pd.read_pickle("model_df.pkl")

short_embeddings = np.load("short_description_embeddings.npy")
short_description_appids = np.load("short_description_appids.npy")

print(model_df.shape)
print(short_embeddings.shape)

(27075, 373)
(27334, 384)


In [2]:
# App IDs that exist in both systems
common_appids = np.intersect1d(
    model_df["appid"].values,
    short_description_appids
)

print("Tag games:", len(model_df))
print("Semantic games:", len(short_description_appids))
print("Common games:", len(common_appids))

Tag games: 27075
Semantic games: 27334
Common games: 27075


In [3]:
# Create a mapping: appid -> embedding index
embedding_index_map = {
    appid: index
    for index, appid in enumerate(short_description_appids)
}

# Get semantic embeddings in EXACTLY the same order as model_df
aligned_embedding_indices = [
    embedding_index_map[appid]
    for appid in model_df["appid"]
]

aligned_short_embeddings = short_embeddings[aligned_embedding_indices]

print("model_df shape:", model_df.shape)
print("Aligned embeddings shape:", aligned_short_embeddings.shape)

model_df shape: (27075, 373)
Aligned embeddings shape: (27075, 384)


In [4]:
from sklearn.preprocessing import normalize

# Everything after appid and name is a tag feature
tag_columns = [
    column for column in model_df.columns
    if column not in ["appid", "name"]
]

# Extract tag matrix
tag_matrix = model_df[tag_columns].values.astype(np.float32)

# Normalize each game's tag vector
normalized_tag_matrix = normalize(tag_matrix, norm="l2")

print("Tag matrix shape:", normalized_tag_matrix.shape)
print("Number of tag features:", len(tag_columns))

Tag matrix shape: (27075, 371)
Number of tag features: 371


In [6]:
from sklearn.preprocessing import normalize
import numpy as np

# Remove non-feature columns
tag_columns = [
    col for col in model_df.columns
    if col not in ["appid", "name"]
]

# Extract tag values
X = model_df[tag_columns].values.astype(np.float32)

# Normalize each game's tag vector
X_normalized = normalize(X, norm="l2")

print("X shape:", X.shape)
print("X_normalized shape:", X_normalized.shape)

X shape: (27075, 371)
X_normalized shape: (27075, 371)


In [7]:
print(X_normalized.shape)
print(aligned_short_embeddings.shape)

(27075, 371)
(27075, 384)


In [21]:
def find_tag_similar_games(game_name, top_n=10):

    matches = model_df[
        model_df["name"].str.lower() == game_name.lower()
    ]

    if matches.empty:
        print(f"Game not found: {game_name}")
        return

    game_index = matches.index[0]

    # Tag cosine similarity
    tag_scores = X_normalized @ X_normalized[game_index]

    # Sort highest to lowest
    similar_indices = np.argsort(tag_scores)[::-1]

    # Remove the game itself
    similar_indices = [
        idx for idx in similar_indices
        if idx != game_index
    ][:top_n]

    results = model_df.iloc[similar_indices][["name", "appid"]].copy()
    results["tag_score"] = tag_scores[similar_indices]

    return results

In [22]:
def find_semantic_similar_games(game_name, top_n=10):

    matches = model_df[
        model_df["name"].str.lower() == game_name.lower()
    ]

    if matches.empty:
        print(f"Game not found: {game_name}")
        return

    game_index = matches.index[0]

    # Semantic cosine similarity
    semantic_scores = (
        aligned_short_embeddings
        @ aligned_short_embeddings[game_index]
    )

    # Sort highest to lowest
    similar_indices = np.argsort(semantic_scores)[::-1]

    # Remove the game itself
    similar_indices = [
        idx for idx in similar_indices
        if idx != game_index
    ][:top_n]

    results = model_df.iloc[similar_indices][["name", "appid"]].copy()
    results["semantic_score"] = semantic_scores[similar_indices]

    return results

In [8]:
def find_hybrid_similar_games(
    game_name,
    top_n=10,
    tag_weight=0.8,
    semantic_weight=0.2
):

    # Find the game
    matches = model_df[
        model_df["name"].str.lower() == game_name.lower()
    ]

    if matches.empty:
        print(f"Game not found: {game_name}")
        return

    game_index = matches.index[0]

    # Tag similarity
    tag_scores = X_normalized @ X_normalized[game_index]

    # Semantic similarity
    semantic_scores = (
        aligned_short_embeddings
        @ aligned_short_embeddings[game_index]
    )

    # Hybrid score
    hybrid_scores = (
        tag_weight * tag_scores
        + semantic_weight * semantic_scores
    )

    # Sort by hybrid similarity
    similar_indices = np.argsort(hybrid_scores)[::-1]

    # Remove the query game itself
    similar_indices = [
        idx for idx in similar_indices
        if idx != game_index
    ][:top_n]

    # Results
    results = model_df.iloc[
        similar_indices
    ][["name", "appid"]].copy()

    results["tag_score"] = tag_scores[similar_indices]
    results["semantic_score"] = semantic_scores[similar_indices]
    results["hybrid_score"] = hybrid_scores[similar_indices]

    return results

In [17]:
def search_games(query, top_n=10):

    query = query.lower().strip()

    matches = model_df[
        model_df["name"]
        .str.lower()
        .str.contains(query, na=False)
    ][["appid", "name"]]

    return matches.head(top_n)

In [18]:
search_games("deus ex")

,appid,name
177,6910,Deus Ex: Game of the Year Edition
178,6920,Deus Ex: Invisible War
1755,238010,Deus Ex: Human Revolution - Director's Cut
2125,258180,Deus Ex: The Fall
4332,337000,Deus Ex: Mankind Divided
5004,353610,DEUS EX MACHINA 2
10244,508910,"Deus Ex Machina, Game of the Year, 30th Annive..."
10907,526180,Deus Ex: Mankind Divided™ - VR Experience
11991,555450,Deus Ex: Breach™


In [24]:
find_tag_similar_games("Dishonored")

,name,appid,tag_score
14034,Dishonored®: Death of the Outsider™,614570,0.861641
6847,Dishonored 2,403640,0.853640
1778,Thief,239160,0.840586
179,Thief: Deadly Shadows,6980,0.805068
1373,Thief™ II: The Metal Age,211740,0.769379
1813,Styx: Master of Shadows,242640,0.738144
1372,Thief™ Gold,211600,0.718499
1899,Hitman: Contracts,247430,0.718003
1214,Hitman: Absolution™,203140,0.678026
173,Hitman: Blood Money,6860,0.672366


In [28]:
find_semantic_similar_games("Dishonored")

,name,appid,semantic_score
6847,Dishonored 2,403640,0.758104
14034,Dishonored®: Death of the Outsider™,614570,0.673858
23228,BrutalAliens,886600,0.477815
4204,Cults and Daggers,333350,0.458260
2911,DubWars,290000,0.456109
9434,Bastard Bonds,486720,0.452216
5131,Aerannis,356580,0.445653
22219,The Cooking Game VR,857180,0.445273
2396,Satellite Reign,268870,0.444557
25492,Chop is dish,974370,0.441291


In [20]:
find_hybrid_similar_games(
    "Dishonored",
    top_n=10
)

,name,appid,tag_score,semantic_score,hybrid_score
6847,Dishonored 2,403640,0.853640,0.758104,0.834532
14034,Dishonored®: Death of the Outsider™,614570,0.861641,0.673858,0.824085
1778,Thief,239160,0.840586,0.083399,0.689149
179,Thief: Deadly Shadows,6980,0.805068,0.165973,0.677249
1373,Thief™ II: The Metal Age,211740,0.769379,0.230107,0.661525
1813,Styx: Master of Shadows,242640,0.738144,0.302964,0.651108
1372,Thief™ Gold,211600,0.718499,0.324728,0.639745
1899,Hitman: Contracts,247430,0.718003,0.309354,0.636273
172,Hitman 2: Silent Assassin,6850,0.658654,0.368193,0.600562
1214,Hitman: Absolution™,203140,0.678026,0.250759,0.592573
